In [2]:
import pandas as pd #data manipulation package to import for analysis
import numpy as np #package for multidimensional array computations
import matplotlib.pyplot as plt #library for plotting (matlab)
import datetime #supplies classes for manipulating date and time

In [4]:
pip install getFamaFrenchFactors

Note: you may need to restart the kernel to use updated packages.


In [5]:
import getFamaFrenchFactors as gff #tool to scrape fama french library for small, value, and market for last 100 years
from pandas_datareader.data import DataReader #package to read data

In [6]:
end=datetime.datetime.today() #sets end as today for data

In [15]:
#the following lines get the last 5 years of monthly stock price data for a given stock (Macy's) and sets it into a dataFrame

macyData=DataReader('M','yahoo','2016-07-01',end)['Adj Close'] 
macyDataF=macydata.resample('M').last()
macyDataF=macydataF.to_frame()
macyDataF['Year']=macyDataF.index.year
macyDataF['Month']=macyDataF.index.month
macyDataF.rename(columns={'Adj Close':'M'},inplace=True)


RemoteDataError: Unable to read URL: https://finance.yahoo.com/quote/M/history?period1=1467363600&period2=1626080399&interval=1d&frequency=1d&filter=history
Response Text:
b'<!DOCTYPE html>\n  <html lang="en-us"><head>\n  <meta http-equiv="content-type" content="text/html; charset=UTF-8">\n      <meta charset="utf-8">\n      <title>Yahoo</title>\n      <meta name="viewport" content="width=device-width,initial-scale=1,minimal-ui">\n      <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">\n      <style>\n  html {\n      height: 100%;\n  }\n  body {\n      background: #fafafc url(https://s.yimg.com/nn/img/sad-panda-201402200631.png) 50% 50%;\n      background-size: cover;\n      height: 100%;\n      text-align: center;\n      font: 300 18px "helvetica neue", helvetica, verdana, tahoma, arial, sans-serif;\n  }\n  table {\n      height: 100%;\n      width: 100%;\n      table-layout: fixed;\n      border-collapse: collapse;\n      border-spacing: 0;\n      border: none;\n  }\n  h

In [11]:
#the following lines get the past 5 years of data from the fama french library

FF3data=gff.famaFrench3Factor(frequency='m')
FF3data=FF3data.tail(60)
FF3data['Year']=FF3data['date_ff_factors'].dt.year

FF3data['Month']=FF3data['date_ff_factors'].dt.month



In [17]:
#the following lines merge the data into a single table

datanow=pd.merge(macyDataF,FF3data,left_on=['Year','Month'],right_on=['Year','Month'])

datanow.sort_values(['date_ff_factors'],axis=0,ascending=False,inplace=True)
datanow['rM']=datanow['M']/datanow.M.shift(-1)-1 #dividing by yesterdays (minus 1)'s price
datanow.dropna(subset=['rM'],inplace=True)
datanow['rMkt']=datanow['Mkt-RF']+datanow['RF']

plt.scatter(datanow['rMkt'],datanow['M'])

m, b = np.polyfit(datanow['rMkt'],datanow['rM'],1)
plt.plot(datanow['rMkt'],b+m*datanow['rMkt'],'r')

NameError: name 'macyDataF' is not defined

In [18]:

import statsmodels.api as sm
y=datanow['rM']-datanow['RF']
x=datanow['Mkt-RF']
x=sm.add_constant(x)

mod=sm.OLS(y,x)
res=mod.fit()
res.summary()


NameError: name 'datanow' is not defined